# 🔬 Colab GPU: 3D VisReg JEPA + Multi-Scale FPN Downstream Fine-Tuning
### Part of the BraTS 3D Volumetric Multimodal JEPA Research Suite

Fine-tunes the **SSL-pretrained 3D VisReg JEPA encoder** (from `colab_a100_visreg_pretrain.ipynb` or Kaggle Notebook 01) with the hierarchical multi-scale 3D FPN decoder on a **Google Colab GPU** (L4 / A100).

Runs the complete empirical evaluation battery:
1. **Full-data held-out test evaluation** ($N=242$ volumes, Dice WT/TC/ET, HD95, TTA)
2. **Low-data label efficiency benchmark** across 6 annotation fractions ($1\%$, $5\%$, $10\%$, $25\%$, $50\%$, $100\%$)
3. **Out-of-Distribution (OOD) scanner shift robustness** (physical Rician noise, quadratic $B_1$ field bias, missing sequences)
4. **Direct export to Google Drive** (`visreg_multiscale_outputs.zip`)

> **Protocol: v2 multiregion (BraTS 2024 WT/TC/ET).** 5-class outputs (bg/NCR/ED/NET/ET); eval emits one row per `(model, region)`. Consistent with Kaggle suite `07_finetune_visreg_multiscale_3d.ipynb`.


## 1. Hardware, CUDA & Google Drive Mounting
Verifies GPU accelerator and mounts Google Drive for persistent storage of fine-tuned weights, logs, and evaluation metrics.


In [ ]:
import datetime
import time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected. Please navigate to Runtime -> Change runtime type -> GPU (T4, L4, or A100)!")

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
thesis_dir = Path("/content/drive/MyDrive/thesis")
if (thesis_dir / "downstream" / "multiscale").exists() and not (thesis_dir / "downstream" / "visreg_multiscale").exists():
    output_dir = thesis_dir / "downstream" / "multiscale"
else:
    output_dir = thesis_dir / "downstream" / "visreg_multiscale"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Fine-tuned weights and evaluation reports will stream to Google Drive:\n  {output_dir}")


## 2. Experiment Configuration & Reproducibility Parameters
Unified configuration block identical to the Kaggle benchmark suite, with optimized batch size ($B=4$) and patch size ($8^3$ or $16^3$).


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 2          # Parallel CPU data loading workers (Colab provides multi-vCPU allocation)
PATCH_SIZE = 8           # Patch size: 8 for high-res 8³ model (4,096 tokens); 16 for standard (512 tokens)
BATCH_SIZE = 4           # Volumetric batch size for downstream fine-tuning (Colab GPU allows B=4; T4 uses B=2)
EPOCHS = 60              # Downstream fine-tuning epochs
LOW_DATA_EPOCHS = 15     # Low-data label efficiency evaluation epochs
LEARNING_RATE = 3e-4     # Base AdamW learning rate
WEIGHT_DECAY = 1e-4      # AdamW weight decay


## 3. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")


## 4. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/content/tumor_segmentation_3d")

# Clean up incomplete clone if needed
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

if not (work_dir / "src" / "brats_jepa_3d").exists():
    print(f"Cloning codebase from: {REPO_URL} ...")
    !git clone {REPO_URL} {work_dir}

os.chdir(str(work_dir))
%cd {work_dir}

src_dir = work_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"

!git -C {work_dir} checkout main
!git -C {work_dir} pull --ff-only origin main
assert (Path(work_dir) / "src/brats_jepa_3d/metrics/regions.py").exists(), \
    "Branch checkout failed: regions.py missing — aborting."
print("On branch main:", end=" ")
!git -C {work_dir} log -1 --oneline


## Dataset Ingestion & Discovery
Extracts the preprocessed BraTS 3D dataset from Google Drive (`My Drive/thesis/data/`) into Colab fast local NVMe storage (`/content/data/`) and validates data integrity.

> **Directory Structure**: The `.zip` extracts to root folder `brats_gli_3d_v2/` containing `metadata.csv` and multi-modal `.npz` volumes.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

# 1. Unpack dataset from Google Drive (fast local NVMe extraction)
thesis_dir = Path("/content/drive/MyDrive/thesis")
data_drive_dir = thesis_dir / "data"
target_data_dir = Path("/content/data/brats_gli_3d_v2")

if not (target_data_dir / "metadata.csv").exists() and not Path("/content/data/metadata.csv").exists():
    zip_candidates = []
    if data_drive_dir.exists():
        zip_candidates = sorted(data_drive_dir.glob("*.zip"))
    if not zip_candidates:
        zip_candidates = sorted(Path("/content/drive/MyDrive").glob("*brats*.zip"))

    if zip_candidates:
        dataset_zip = zip_candidates[0]
        print(f"✓ Found dataset archive on Google Drive: {dataset_zip}")
        print("  Unpacking to /content/data/ (fast local NVMe SSD) ...")
        !mkdir -p /content/data
        !unzip -q "{dataset_zip}" -d /content/data/
        print("✓ Dataset extracted successfully.")
    elif data_drive_dir.exists() and any((data_drive_dir / sub / "metadata.csv").exists() for sub in ["brats_gli_3d_v2", "."]):
        print(f"✓ Found pre-extracted dataset in {data_drive_dir}.")
        source_folder = data_drive_dir / "brats_gli_3d_v2" if (data_drive_dir / "brats_gli_3d_v2").exists() else data_drive_dir
        !mkdir -p /content/data
        !cp -rs "{source_folder}" /content/data/brats_gli_3d_v2
    else:
        raise FileNotFoundError(
            f"❌ No dataset .zip found in {data_drive_dir}!\n"
            f"Please upload your dataset .zip (containing 'brats_gli_3d_v2') into 'My Drive/thesis/data/'."
        )

# 2. Discover dataset directory (points to brats_gli_3d_v2)
candidates = [
    Path("/content/data/brats_gli_3d_v2"),
    Path("/content/data"),
    Path("/content/brats_gli_3d_v2"),
]
data_dir = next((c for c in candidates if (c / "metadata.csv").exists()), None)
if data_dir is None:
    matches = list(Path("/content/data").glob("**/metadata.csv"))
    if matches:
        data_dir = matches[0].parent
    else:
        raise FileNotFoundError("❌ metadata.csv not found after extraction in /content/data/!")

meta_path = data_dir / "metadata.csv"
os.environ["BRATS3D_DATA_DIR"] = str(data_dir)

print(f"\n=== DATASET DISCOVERY ===")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())
    assert "has_et" in df.columns, "metadata.csv lacks has_et — attach v2 integer mask dataset."

    ds = BraTS3DDataset(data_dir=data_dir, split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    raise FileNotFoundError(f"Metadata CSV not found at {meta_path}. Please check your dataset zip.")


## 6. Pre-trained Checkpoint Intake (Fail-Loud)
Intakes the pre-trained 3D VisReg JEPA encoder checkpoint from Google Drive (`visreg_jepa_a100/visreg_jepa_3d_best.pt` or `visreg_pretrain_outputs_a100.zip`). Fails loud if absent so we never accidentally benchmark random initialization.


In [ ]:
from pathlib import Path
from brats_jepa_3d.config import CHECKPOINTS_DIR

# Direct checkpoint intake: checks active outputs, then Google Drive thesis folder
candidates = [
    CHECKPOINTS_DIR / "visreg_jepa_3d_best.pt",
    Path("/content/drive/MyDrive/thesis/pretrain model/checkpoints/visreg_jepa_3d_best.pt"),
    Path("/content/drive/MyDrive/thesis/pretrain model/visreg_jepa_3d_best.pt"),
    Path("/content/tumor_segmentation_3d/outputs/checkpoints/visreg_jepa_3d_best.pt"),
]

pretrained_ckpt = next((str(p) for p in candidates if p.exists()), None)
assert pretrained_ckpt, (
    f"❌ Pre-trained checkpoint not found! Checked locations:\n" +
    "\n".join(f"  - {c}" for c in candidates)
)
print(f"✓ Using pre-trained encoder checkpoint: {pretrained_ckpt}")


## 7. Downstream Volumetric Fine-Tuning with Multi-Scale 3D FPN
Couples the pre-trained 3D VisReg encoder with a hierarchical 3D Feature Pyramid Network (FPN) decoder fusing representations across scales with deep supervision and AMP.


In [ ]:
# Fallback defaults if configuration cell was skipped
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
BATCH_SIZE = globals().get('BATCH_SIZE', 4)
EPOCHS = globals().get('EPOCHS', 60)
SEED = globals().get('SEED', 42)
NUM_WORKERS = globals().get('NUM_WORKERS', 2)
LEARNING_RATE = globals().get('LEARNING_RATE', 3e-4)
WEIGHT_DECAY = globals().get('WEIGHT_DECAY', 1e-4)

!python scripts/train_downstream_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --pretrained_checkpoint "{pretrained_ckpt}" \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate {LEARNING_RATE} \
    --weight_decay {WEIGHT_DECAY} \
    --deep_supervision \
    --amp


## 8. Full-Data Held-Out Test Split Evaluation
Evaluates the trained 3D VisReg JEPA segmentation model across all $242$ held-out test volumes with Test-Time Augmentation (TTA).


In [ ]:
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
BATCH_SIZE = globals().get('BATCH_SIZE', 4)
SEED = globals().get('SEED', 42)
NUM_WORKERS = globals().get('NUM_WORKERS', 2)

!python scripts/evaluate_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --seed {SEED} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp \
    --tta


## 9. Low-Data Volumetric Label Efficiency Benchmark
Evaluates fine-tuning performance across annotation budgets: $1\%$ ($11$ vols), $5\%$ ($57$ vols), $10\%$ ($114$ vols), $25\%$ ($286$ vols), $50\%$ ($572$ vols), and $100\%$ ($1,144$ vols).


In [ ]:
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
BATCH_SIZE = globals().get('BATCH_SIZE', 4)
LOW_DATA_EPOCHS = globals().get('LOW_DATA_EPOCHS', 15)
SEED = globals().get('SEED', 42)
NUM_WORKERS = globals().get('NUM_WORKERS', 2)

!python scripts/evaluate_low_data_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --seed {SEED} \
    --epochs {LOW_DATA_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp


## 10. Out-of-Distribution (OOD) Scanner Shift Robustness
Evaluates the multiscale-FPN 3D VisReg JEPA under physical 3D Rician scanner noise ($\sigma=0.08$), quadratic RF $B_1$ field bias, and missing sequence triage (T1c-only, FLAIR-only).


In [ ]:
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
SEED = globals().get('SEED', 42)
NUM_WORKERS = globals().get('NUM_WORKERS', 2)

!python scripts/evaluate_ood_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --seed {SEED} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp


## 11. Export & Package Artifacts to Google Drive
Packages multiscale-finetuned weights, training logs, and metric reports directly into Google Drive for 1-click archiving.


In [ ]:
import datetime
import time
from pathlib import Path
from brats_jepa_3d.utils import export_artifacts

zip_dest = output_dir / "visreg_multiscale_outputs.zip"
export_res = export_artifacts(
    export_name="visreg_multiscale_outputs",
    export_dir=Path("/content/export_visreg_multiscale"),
    zip_path=zip_dest,
    source_dir=output_dir,
    model_prefix="multiscale",
    verbose=True,
)

NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
print(f"✓ Checkpoint and evaluation package saved to Google Drive:\n  {zip_dest}")
